# ใบงานที่ 2: Data Preprocessing
## Dataset: UNESCO World Heritage Sites - 50 years (1978–2026)
**แหล่งข้อมูล:** https://www.kaggle.com/datasets/darkmatternet/unesco-world-heritage-sites-dataset-19782026

ใบงานนี้ประกอบด้วย 4 ส่วนหลัก ตามคำสั่งงาน:

- **LAB1: Dataset Exploration** — โหลดข้อมูล ตรวจสอบขนาด ชนิดข้อมูล สถิติเบื้องต้น ค่าที่หายไป ข้อมูลซ้ำ และการกระจายของคลาส
- **LAB2: Data Visualization** — Histogram และ Correlation Heatmap
- **Part 3: Data Cleaning** — จัดการ Missing Value, ลบข้อมูลซ้ำ, แก้ไขข้อมูลผิดปกติ, แปลงชนิดข้อมูล และเปรียบเทียบ Mean vs Median
- **Part 4: Feature Engineering** — Label Encoding และ One-Hot Encoding

> **หมายเหตุสำคัญ:** เนื่องจากชื่อคอลัมน์จริงของไฟล์ CSV อาจแตกต่างกันไปตามเวอร์ชันที่โหลดจาก Kaggle
> โค้ดในโน้ตบุ๊กนี้ถูกออกแบบให้ **ตรวจจับคอลัมน์อัตโนมัติ** (เช่น หาคอลัมน์ปี, พื้นที่, ประเภท, ประเทศ)
> แทนการเขียนชื่อคอลัมน์ตายตัว หากตรวจไม่พบให้ดูรายชื่อคอลัมน์จริงจาก `df.columns` ที่พิมพ์ไว้ในเซลล์ LAB1
> แล้วปรับตัวแปรใน `COLUMN OVERRIDE` (เซลล์ถัดไปจากการโหลดข้อมูล) ให้ตรงกับไฟล์ของคุณ

## 0. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
plt.rcParams["figure.dpi"] = 110


## LAB1: Dataset Exploration
### 1.1 Load Dataset

วิธีโหลดข้อมูล มี 2 ทางเลือก:

1. **ถ้ารันบน Kaggle Notebook** และแนบ (Add Data) dataset นี้ไว้แล้ว ให้ใช้ path แบบ
   `/kaggle/input/unesco-world-heritage-sites-dataset-19782026/<ชื่อไฟล์.csv>`
2. **ถ้ารันในเครื่อง/Colab** ให้ดาวน์โหลดไฟล์ CSV จาก Kaggle มาไว้โฟลเดอร์เดียวกับโน้ตบุ๊ก แล้วแก้ `CSV_PATH` ด้านล่าง

In [ ]:
# แก้ path ให้ตรงกับตำแหน่งไฟล์ของคุณ
CSV_PATH = "unesco_world_heritage_sites.csv"  # <-- เปลี่ยนเป็นชื่อไฟล์จริง

import glob, os

def smart_load(path):
    # ถ้าไม่พบไฟล์ตามชื่อที่ระบุ ให้ลองค้นหาไฟล์ .csv ทั้งหมดในโฟลเดอร์ปัจจุบันและ /kaggle/input
    candidates = [path]
    candidates += glob.glob("*.csv")
    candidates += glob.glob("/kaggle/input/**/*.csv", recursive=True)
    for c in candidates:
        if os.path.exists(c):
            print(f"Loading: {c}")
            return pd.read_csv(c)
    raise FileNotFoundError(
        "ไม่พบไฟล์ CSV กรุณาดาวน์โหลด dataset จาก Kaggle แล้ววางไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊กนี้ "
        "หรือแก้ตัวแปร CSV_PATH ให้ตรงกับตำแหน่งไฟล์จริง"
    )

df = smart_load(CSV_PATH)
df_raw = df.copy()  # เก็บสำเนาข้อมูลดิบไว้เผื่อเทียบผลลัพธ์ก่อน/หลัง cleaning
df.head()


In [ ]:
# ----- COLUMN OVERRIDE -----
# หากการตรวจจับคอลัมน์อัตโนมัติด้านล่างไม่ถูกต้อง ให้พิมพ์ชื่อคอลัมน์จริงที่นี่ (หรือปล่อยเป็น None เพื่อให้ระบบเดาอัตโนมัติ)
COL_YEAR     = None   # เช่น "date_inscribed" หรือ "year_inscribed"
COL_AREA     = None   # เช่น "area_hectares"
COL_CATEGORY = None   # เช่น "category" (Cultural / Natural / Mixed)
COL_COUNTRY  = None   # เช่น "country" หรือ "states_party"
COL_REGION   = None   # เช่น "region" หรือ "unesco_region"

def find_column(df, override, keywords):
    if override is not None and override in df.columns:
        return override
    for col in df.columns:
        low = col.lower()
        if any(k in low for k in keywords):
            return col
    return None

COL_YEAR     = find_column(df, COL_YEAR,     ["year", "date_inscribed", "inscribed"])
COL_AREA     = find_column(df, COL_AREA,     ["area", "hectare", "size"])
COL_CATEGORY = find_column(df, COL_CATEGORY, ["categor", "type_of_site", "heritage_type"])
COL_COUNTRY  = find_column(df, COL_COUNTRY,  ["country", "state", "party", "nation"])
COL_REGION   = find_column(df, COL_REGION,   ["region", "continent"])

print("Detected columns ->")
print(f"  Year column     : {COL_YEAR}")
print(f"  Area column     : {COL_AREA}")
print(f"  Category column : {COL_CATEGORY}")
print(f"  Country column  : {COL_COUNTRY}")
print(f"  Region column   : {COL_REGION}")
print("\nAll columns in dataset:")
print(list(df.columns))


### 1.2 Display Shape

In [ ]:
print(f"จำนวนแถว (rows)   : {df.shape[0]}")
print(f"จำนวนคอลัมน์ (cols): {df.shape[1]}")
df.shape


### 1.3 Display Data Types

In [ ]:
df.dtypes.to_frame(name="dtype")


### 1.4 Display Summary Statistics

In [ ]:
print("สถิติเชิงพรรณนา (ตัวเลข):")
display(df.describe(include=[np.number]).T)

print("\nสถิติเชิงพรรณนา (ข้อความ/หมวดหมู่):")
display(df.describe(include=["object"]).T)


### 1.5 Display Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct(%)": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values(
    "missing_count", ascending=False
)
print(f"จำนวนคอลัมน์ที่มีค่า missing: {len(missing_summary)} จาก {df.shape[1]} คอลัมน์")
missing_summary


In [ ]:
# แสดงเป็นภาพเพื่อดูภาพรวมง่ายขึ้น
plt.figure(figsize=(8, max(3, len(missing_summary) * 0.4)))
if len(missing_summary) > 0:
    sns.barplot(x=missing_summary["missing_pct(%)"], y=missing_summary.index, color="#008ABC")
    plt.xlabel("Missing (%)")
    plt.title("Missing Values by Column")
else:
    plt.text(0.5, 0.5, "ไม่มีค่า Missing ในข้อมูลนี้", ha="center", va="center")
    plt.axis("off")
plt.tight_layout()
plt.show()


### 1.6 Display Duplicate Records

In [ ]:
dup_count = df.duplicated().sum()
print(f"จำนวนแถวที่ซ้ำกันทั้งหมด (exact duplicate rows): {dup_count}")
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(20)


In [ ]:
# ตรวจสอบข้อมูลซ้ำจาก "ชื่อสถานที่" ด้วย (อาจซ้ำทั้งแถวไม่เท่ากันแต่ชื่อซ้ำกัน)
name_col = find_column(df, None, ["name", "site"])
if name_col:
    dup_by_name = df[df.duplicated(subset=[name_col], keep=False)]
    print(f"คอลัมน์ที่ใช้ตรวจสอบชื่อ: {name_col}")
    print(f"จำนวนแถวที่ชื่อซ้ำกัน: {len(dup_by_name)}")
    dup_by_name.sort_values(by=name_col).head(20)
else:
    print("ไม่พบคอลัมน์ชื่อสถานที่โดยอัตโนมัติ")


### 1.7 Display Class Distribution

ใช้คอลัมน์ประเภทมรดกโลก (`Cultural` / `Natural` / `Mixed`) หากตรวจพบ มิฉะนั้นจะเลือกคอลัมน์ข้อความที่มีจำนวนค่าไม่ซ้ำน้อยที่สุด (low-cardinality) เป็นตัวแทน class

In [ ]:
target_col = COL_CATEGORY
if target_col is None:
    obj_cols = df.select_dtypes(include="object").columns
    nunique = df[obj_cols].nunique().sort_values()
    nunique = nunique[nunique.between(2, 15)]  # เลือกคอลัมน์ที่เหมาะกับการเป็น "class" (ไม่มากเกินไป)
    target_col = nunique.index[0] if len(nunique) > 0 else None

if target_col:
    print(f"คอลัมน์ที่ใช้แสดง Class Distribution: {target_col}\n")
    dist = df[target_col].value_counts(dropna=False)
    dist_pct = (dist / len(df) * 100).round(2)
    display(pd.DataFrame({"count": dist, "pct(%)": dist_pct}))

    plt.figure(figsize=(7, 4))
    sns.barplot(x=dist.values, y=dist.index.astype(str), color="#E67E22")
    plt.xlabel("Count")
    plt.title(f"Class Distribution: {target_col}")
    plt.tight_layout()
    plt.show()
else:
    print("ไม่พบคอลัมน์ที่เหมาะสมสำหรับแสดง class distribution กรุณากำหนด COL_CATEGORY ด้วยตนเอง")


## LAB2: Data Visualization
### 2.1 Histogram

แสดง histogram ของทุกคอลัมน์ตัวเลข เพื่อดูการกระจายตัวและสังเกต outlier เบื้องต้น

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print(f"คอลัมน์ตัวเลขที่พบ: {numeric_cols}")

if numeric_cols:
    n = len(numeric_cols)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, col in enumerate(numeric_cols):
        axes[i].hist(df[col].dropna(), bins=30, color="#2980B9", edgecolor="white")
        axes[i].set_title(col)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("ไม่พบคอลัมน์ตัวเลขในข้อมูล")


### 2.2 Correlation Heatmap

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(1.2 * len(numeric_cols) + 2, 1.0 * len(numeric_cols) + 2))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
                linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title("Correlation Heatmap (Numeric Features)")
    plt.tight_layout()
    plt.show()
else:
    print("มีคอลัมน์ตัวเลขไม่พอ (ต้องมีอย่างน้อย 2 คอลัมน์) สำหรับทำ correlation heatmap")


## Part 3: Data Cleaning

ในส่วนนี้จะทำการทำความสะอาดข้อมูลทีละขั้นตอน โดยเก็บผลลัพธ์ไว้ใน `df_clean`

In [ ]:
df_clean = df.copy()


### 3.1 Missing Value Handling

กลยุทธ์ที่ใช้:
- คอลัมน์ตัวเลข → เติมด้วยค่ากลาง (median) เนื่องจากข้อมูลพื้นที่ (area) ของแหล่งมรดกโลกมักมีการกระจายแบบเบ้ (skewed) และมี outlier สูง median จึงทนทานกว่า mean
- คอลัมน์ข้อความ/หมวดหมู่ → เติมด้วยค่าที่พบมากที่สุด (mode) หรือ `"Unknown"` หากไม่มีค่าเด่นชัด

In [ ]:
num_missing_before = df_clean.select_dtypes(include=np.number).isnull().sum()
print("Missing (numeric) ก่อนเติม:\n", num_missing_before[num_missing_before > 0])

for col in df_clean.select_dtypes(include=np.number).columns:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)

for col in df_clean.select_dtypes(include="object").columns:
    if df_clean[col].isnull().sum() > 0:
        mode_series = df_clean[col].mode(dropna=True)
        fill_val = mode_series[0] if len(mode_series) > 0 else "Unknown"
        df_clean[col] = df_clean[col].fillna(fill_val)

print("\nMissing values หลังการเติมค่า:")
print(df_clean.isnull().sum().sum(), "ค่าที่ยังหายไปทั้งหมด")


### 3.2 Duplicate Removal

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
after = len(df_clean)
print(f"จำนวนแถวก่อนลบข้อมูลซ้ำ: {before}")
print(f"จำนวนแถวหลังลบข้อมูลซ้ำ: {after}")
print(f"ลบไปทั้งหมด: {before - after} แถว")


### 3.3 Incorrect Data Correction

ตรวจสอบและแก้ไขข้อมูลที่ผิดปกติ เช่น:
- ปีที่จารึก (inscribed year) ต้องอยู่ในช่วง 1978–2026 (ตามชื่อ dataset)
- พื้นที่ (area) ต้องไม่เป็นค่าติดลบ
- ข้อความมีช่องว่างเกิน หรือรูปแบบตัวพิมพ์ใหญ่/เล็กไม่สอดคล้องกัน (เช่น "italy" vs "Italy")

In [ ]:
# 3.3.1 ปีที่ไม่สมเหตุสมผล
if COL_YEAR and COL_YEAR in df_clean.columns:
    year_numeric = pd.to_numeric(df_clean[COL_YEAR], errors="coerce").astype(float)
    invalid_year_mask = (year_numeric < 1978) | (year_numeric > 2026)
    n_invalid = invalid_year_mask.sum()
    print(f"จำนวนแถวที่ปี '{COL_YEAR}' ผิดปกติ (นอกช่วง 1978-2026): {n_invalid}")
    # แก้ไข: แทนที่ปีผิดปกติด้วยค่า median ของปีที่ถูกต้อง
    valid_median_year = year_numeric[~invalid_year_mask].median()
    year_numeric[invalid_year_mask] = valid_median_year
    df_clean[COL_YEAR] = year_numeric

# 3.3.2 พื้นที่ติดลบ
if COL_AREA and COL_AREA in df_clean.columns:
    area_numeric = pd.to_numeric(df_clean[COL_AREA], errors="coerce").astype(float)
    negative_mask = area_numeric < 0
    print(f"จำนวนแถวที่พื้นที่ '{COL_AREA}' เป็นค่าติดลบ: {negative_mask.sum()}")
    area_numeric[negative_mask] = area_numeric[~negative_mask].median()
    df_clean[COL_AREA] = area_numeric

# 3.3.3 ข้อความ: strip whitespace + ปรับรูปแบบตัวพิมพ์ให้สอดคล้อง (title case) สำหรับคอลัมน์ประเทศ/ภูมิภาค
for col in [COL_COUNTRY, COL_REGION, COL_CATEGORY]:
    if col and col in df_clean.columns and df_clean[col].dtype == object:
        before_unique = df_clean[col].nunique()
        df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
        after_unique = df_clean[col].nunique()
        print(f"คอลัมน์ '{col}': จำนวนค่าไม่ซ้ำก่อนแก้ไข = {before_unique}, หลังแก้ไข = {after_unique}")


### 3.4 Data Type Conversion

In [ ]:
# แปลงคอลัมน์ปีให้เป็นจำนวนเต็ม (Int64 รองรับ NaN ได้)
if COL_YEAR and COL_YEAR in df_clean.columns:
    df_clean[COL_YEAR] = pd.to_numeric(df_clean[COL_YEAR], errors="coerce").round().astype("Int64")

# แปลงคอลัมน์พื้นที่ให้เป็นทศนิยม
if COL_AREA and COL_AREA in df_clean.columns:
    df_clean[COL_AREA] = pd.to_numeric(df_clean[COL_AREA], errors="coerce").astype(float)

# แปลงคอลัมน์ข้อความหมวดหมู่ให้เป็น category dtype (ประหยัดหน่วยความจำ + เหมาะกับการทำ ML)
for col in [COL_CATEGORY, COL_COUNTRY, COL_REGION]:
    if col and col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("category")

print("ชนิดข้อมูลหลังแปลง:")
df_clean.dtypes.to_frame(name="dtype")


### 3.5 เปรียบเทียบ Mean vs Median

เปรียบเทียบผลลัพธ์การเติมค่า missing ด้วย mean เทียบกับ median บนคอลัมน์ตัวเลขหลัก (เช่น พื้นที่ของแหล่งมรดกโลก)
เพื่อแสดงว่าเมื่อข้อมูลมี outlier/การกระจายแบบเบ้ ค่า mean จะถูกดึงสูงกว่าค่าจริงส่วนใหญ่ ในขณะที่ median จะทนทานกว่า

In [ ]:
compare_col = COL_AREA if COL_AREA else (numeric_cols[0] if numeric_cols else None)

if compare_col:
    original_series = pd.to_numeric(df_raw[compare_col], errors="coerce")
    mean_val = original_series.mean()
    median_val = original_series.median()

    filled_with_mean = original_series.fillna(mean_val)
    filled_with_median = original_series.fillna(median_val)

    comparison = pd.DataFrame({
        "Statistic": ["mean", "median", "std", "min", "max"],
        "Original (dropna)": [
            original_series.mean(), original_series.median(),
            original_series.std(), original_series.min(), original_series.max()
        ],
        "After fill w/ Mean": [
            filled_with_mean.mean(), filled_with_mean.median(),
            filled_with_mean.std(), filled_with_mean.min(), filled_with_mean.max()
        ],
        "After fill w/ Median": [
            filled_with_median.mean(), filled_with_median.median(),
            filled_with_median.std(), filled_with_median.min(), filled_with_median.max()
        ],
    })
    display(comparison)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(filled_with_mean.dropna(), bins=30, color="#C0392B", edgecolor="white")
    axes[0].axvline(mean_val, color="black", linestyle="--", label=f"mean={mean_val:.1f}")
    axes[0].set_title(f"Filled with Mean: {compare_col}")
    axes[0].legend()

    axes[1].hist(filled_with_median.dropna(), bins=30, color="#27AE60", edgecolor="white")
    axes[1].axvline(median_val, color="black", linestyle="--", label=f"median={median_val:.1f}")
    axes[1].set_title(f"Filled with Median: {compare_col}")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"สรุป: ค่า mean ({mean_val:.2f}) {'สูงกว่า' if mean_val > median_val else 'ต่ำกว่า'} "
          f"ค่า median ({median_val:.2f}) อย่างมีนัยสำคัญ แสดงว่าคอลัมน์นี้มีการกระจายแบบเบ้/มี outlier "
          f"→ การเติมค่า missing ด้วย median จึงเหมาะสมกว่าในกรณีนี้")
else:
    print("ไม่พบคอลัมน์ตัวเลขที่เหมาะสมสำหรับเปรียบเทียบ mean/median")


## Part 4: Feature Engineering
### 4.1 Label Encoding

ใช้กับคอลัมน์ประเภท (Category) ที่มีลำดับ/จำนวนคลาสน้อย เช่น `Cultural / Natural / Mixed`
Label Encoding เหมาะกับโมเดลที่ไม่ไวต่อลำดับตัวเลข (เช่น Tree-based models) หรือกรณีมีข้อมูลเชิงลำดับ (ordinal)

In [ ]:
df_encoded = df_clean.copy()

label_target = COL_CATEGORY
if label_target and label_target in df_encoded.columns:
    le = LabelEncoder()
    df_encoded[f"{label_target}_label_encoded"] = le.fit_transform(df_encoded[label_target].astype(str))
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"Label Encoding บนคอลัมน์: {label_target}")
    print("Mapping:", mapping)
    display(df_encoded[[label_target, f"{label_target}_label_encoded"]].drop_duplicates().reset_index(drop=True))
else:
    print("ไม่พบคอลัมน์ประเภท (category) ที่เหมาะสมสำหรับ Label Encoding กรุณากำหนด COL_CATEGORY เอง")


### 4.2 One-Hot Encoding

ใช้กับคอลัมน์ nominal ที่ไม่มีลำดับ เช่น `region` (ภูมิภาค) เหมาะกับโมเดลเชิงเส้น/ระยะทาง (เช่น Linear Regression, KNN, Neural Network)
ที่ไม่ต้องการให้ตัวเลขของ label สื่อถึง "ลำดับ" หรือ "ระยะห่าง" ที่ไม่มีอยู่จริง

In [ ]:
onehot_target = COL_REGION if COL_REGION else COL_COUNTRY

if onehot_target and onehot_target in df_encoded.columns:
    # จำกัดจำนวนหมวดหมู่ไม่ให้มากเกินไป (one-hot ของคอลัมน์ cardinality สูงอย่าง country จะสร้างคอลัมน์จำนวนมาก)
    n_unique = df_encoded[onehot_target].nunique()
    print(f"คอลัมน์ที่ใช้ One-Hot Encoding: {onehot_target} ({n_unique} หมวดหมู่)")

    onehot = pd.get_dummies(df_encoded[onehot_target], prefix=onehot_target)
    df_encoded = pd.concat([df_encoded, onehot], axis=1)
    display(onehot.head())
    print(f"\nจำนวนคอลัมน์ใหม่ที่เพิ่มจาก One-Hot Encoding: {onehot.shape[1]}")
else:
    print("ไม่พบคอลัมน์ที่เหมาะสมสำหรับ One-Hot Encoding กรุณากำหนด COL_REGION หรือ COL_COUNTRY เอง")


### 4.3 ตรวจสอบผลลัพธ์ข้อมูลสุดท้าย

In [ ]:
print(f"ขนาดข้อมูลก่อนทำ preprocessing : {df_raw.shape}")
print(f"ขนาดข้อมูลหลังทำ preprocessing  : {df_encoded.shape}")
df_encoded.head()


## สรุป (Conclusion)

- ตรวจสอบและทำความเข้าใจโครงสร้างข้อมูล (shape, dtypes, สถิติเบื้องต้น) ก่อนดำเนินการใดๆ
- พบและจัดการค่า Missing Values, ข้อมูลซ้ำ, และข้อมูลผิดปกติ (เช่น ปีนอกช่วง 1978–2026, พื้นที่ติดลบ)
- ใช้ Histogram และ Correlation Heatmap เพื่อทำความเข้าใจการกระจายตัวและความสัมพันธ์ของตัวแปรตัวเลข
- เปรียบเทียบการเติมค่าด้วย Mean และ Median พบว่า Median เหมาะสมกว่าเมื่อข้อมูลมีการกระจายแบบเบ้/มี outlier
- แปลงข้อมูลหมวดหมู่ด้วย Label Encoding (สำหรับ category ที่มีจำนวนคลาสน้อย) และ One-Hot Encoding (สำหรับ nominal feature)
  เพื่อเตรียมข้อมูลให้พร้อมสำหรับการนำไปพัฒนาแบบจำลอง Machine Learning ต่อไป
